# RelCheck — Synthetic Hallucination Test (v2)

**Refactored version** — imports from `relcheck_v2/` package instead of inline code.

**Approach:** Inject known hallucinations into accurate captions and measure whether
RelCheck detects and corrects them.

| Cell | What |
|------|------|
| 1 | Setup: install deps, init API, load models |
| 2 | Load R-Bench data + images |
| 3 | Caption generation |
| 4 | Inject hallucinations from R-Bench GT=no questions |
| 5 | RelCheck detection on corrupted captions |
| 6 | Visual Knowledge Base construction |
| 7 | Full RelCheck correction pipeline |
| 8 | R-POPE LLM-Judge evaluation |

In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
!pip install together Pillow requests transformers rapidfuzz spacy tenacity -q
!python -m spacy download en_core_web_sm -q

import os, json, re, time, random, base64
from pathlib import Path
from collections import Counter, defaultdict
from io import BytesIO
from PIL import Image
from google.colab import drive

# ── Config ──────────────────────────────────────────────────
TOGETHER_API_KEY = ''   # <-- paste your Together.ai key
N_IMAGES         = 20   # start small for validation
CAPTIONER        = 'llava'   # 'blip2' | 'llava' | 'qwen'
RANDOM_SEED      = 42

# Drive paths
DRIVE_IMAGES_DIR = '/content/drive/MyDrive/RelCheck_Data/images'
RBENCH_PATH      = '/content/drive/MyDrive/RelCheck_Data/rbench_data.json'
SAVE_DIR         = '/content/drive/MyDrive/RelCheck_Data/synthetic_test'

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
random.seed(RANDOM_SEED)

# ── Clone/update repo ─────────────────────────────────────
import sys
REPO_DIR = '/content/RelCheck'
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git clone https://github.com/SiddhiPatil06/RelCheck {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
sys.path.insert(0, REPO_DIR)

# ── Initialize relcheck_v2 ────────────────────────────────
from relcheck_v2.api import init_client
from relcheck_v2.config import (
    LLM_MODEL, VLM_MODEL, GDINO_ID,
    GDINO_BOX_THRESHOLD, GDINO_TEXT_THRESHOLD,
    BROAD_CATEGORIES, DESCRIBE_PROMPT,
)
from relcheck_v2.models import get_gdino, get_llava, get_blip2, DEVICE
from relcheck_v2.detection import detect_objects, dedup_detections
from relcheck_v2.verification import verify_triple, classify_relation
from relcheck_v2.kb import build_visual_kb
from relcheck_v2.captioning import (
    caption_image, caption_image_api, caption_image_llava, caption_image_blip2,
)
from relcheck_v2.injection import (
    question_to_statement, parse_question, classify_rel_type,
)
from relcheck_v2.evaluation import rpope_judge, compute_rpope_stats, format_rpope_summary
from relcheck_v2.entity import levenshtein_distance, edit_rate
from relcheck_v2.api import vlm_call, llm_call, encode_b64

# Init Together client
init_client(TOGETHER_API_KEY)

# ── Load models ───────────────────────────────────────────
import torch
print(f'Device: {DEVICE}')

# GroundingDINO (always needed)
gdino_model, gdino_processor = get_gdino()

# Captioner (conditional)
if CAPTIONER == 'llava':
    llava_model, llava_proc = get_llava()
elif CAPTIONER == 'blip2':
    blip2_model, blip2_proc = get_blip2()

print('Setup complete.')

In [ ]:
# ============================================================
# Cell 2 — Load R-Bench Data + Images
# ============================================================
import pathlib

os.makedirs(SAVE_DIR, exist_ok=True)

# ── Download R-Bench if not cached ──
if not os.path.exists(RBENCH_PATH):
    print('R-Bench data not found. Downloading...')
    RBENCH_DIR = '/content/R-Bench'
    if not os.path.exists(RBENCH_DIR):
        !git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}
    DRIVE_FILE_ID = '1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH'
    ANNOTATIONS_ZIP = f'{RBENCH_DIR}/rbench_annotations.zip'
    !gdown --id {DRIVE_FILE_ID} -O {ANNOTATIONS_ZIP} -q
    import zipfile
    with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
        z.extractall(RBENCH_DIR)
    all_jsons = sorted(pathlib.Path(RBENCH_DIR).rglob('*.json'))
    rbench_raw = None
    for f in all_jsons:
        if 'image-level' in f.name.lower() or 'image_level' in f.name.lower():
            with open(f) as fh: data = json.load(fh)
            if isinstance(data, list) and len(data) > 50:
                rbench_raw = data; break
    if rbench_raw is None:
        for f in all_jsons:
            with open(f) as fh: data = json.load(fh)
            if isinstance(data, list) and len(data) > 100:
                rbench_raw = data; break
    rbench_data = {}
    for entry in rbench_raw:
        img = entry.get('image', entry.get('img', ''))
        img_id = os.path.splitext(os.path.basename(img))[0]
        q = str(entry.get('text', entry.get('question', '')))
        a = str(entry.get('label', entry.get('answer', ''))).lower().strip()
        if img_id and q:
            rbench_data.setdefault(img_id, []).append({'question': q, 'answer': a})
    with open(RBENCH_PATH, 'w') as f: json.dump(rbench_data, f)
    print(f'Saved {len(rbench_data)} images, {sum(len(v) for v in rbench_data.values())} questions')
else:
    with open(RBENCH_PATH) as f: rbench_data = json.load(f)
    if isinstance(rbench_data, list):
        _raw = rbench_data
        rbench_data = {}
        for entry in _raw:
            img = entry.get('image', entry.get('img', ''))
            img_id = os.path.splitext(os.path.basename(img))[0]
            q = str(entry.get('text', entry.get('question', '')))
            a = str(entry.get('label', entry.get('answer', ''))).lower().strip()
            if img_id and q:
                rbench_data.setdefault(img_id, []).append({'question': q, 'answer': a})
    print(f'Loaded R-Bench: {len(rbench_data)} images, {sum(len(v) for v in rbench_data.values())} questions')

# ── Load images ──
if not os.path.exists(DRIVE_IMAGES_DIR):
    print(f'Images dir not found: {DRIVE_IMAGES_DIR}')
    print('Please download nocaps images and place in that folder.')
else:
    all_files = list(pathlib.Path(DRIVE_IMAGES_DIR).glob('*.jpg'))
    available = {f.stem: str(f) for f in all_files}
    pool = [(img_id, qs) for img_id, qs in rbench_data.items() if img_id in available]
    random.seed(RANDOM_SEED)
    selected = random.sample(pool, min(N_IMAGES, len(pool)))

    pil_images = {}
    rbench_questions = {}
    for img_id, qs in selected:
        try:
            pil_images[img_id] = Image.open(available[img_id]).convert('RGB')
            rbench_questions[img_id] = qs
        except Exception as e:
            print(f'  [{img_id}] load error: {e}')

    print(f'Loaded {len(pil_images)} images with '
          f'{sum(len(v) for v in rbench_questions.values())} R-Bench questions')

In [ ]:
# ============================================================
# Cell 3 — Caption Generation (incremental)
# ============================================================
CAPTIONS_PATH = f'{SAVE_DIR}/{CAPTIONER}_captions.json'

if os.path.exists(CAPTIONS_PATH):
    with open(CAPTIONS_PATH) as f: captions = json.load(f)
    print(f'Loaded {len(captions)} cached captions from {CAPTIONS_PATH}')
else:
    captions = {}

new_count = 0
for img_id, pil in pil_images.items():
    if img_id in captions:
        continue
    cap = caption_image(pil, captioner=CAPTIONER)
    if cap:
        captions[img_id] = cap
        new_count += 1
        print(f'  [{img_id}] {cap[:80]}')

if new_count > 0:
    with open(CAPTIONS_PATH, 'w') as f: json.dump(captions, f, indent=2)
    print(f'Generated {new_count} new captions (total: {len(captions)})')
else:
    print(f'All {len(captions)} captions already cached.')

In [ ]:
# ============================================================
# Cell 4 — Inject Hallucinations from R-Bench GT=no Questions
# ============================================================
INJECT_PATH = f'{SAVE_DIR}/injected_{CAPTIONER}.json'

if Path(INJECT_PATH).exists():
    with open(INJECT_PATH) as f: injected_data = json.load(f)
    print(f'Loaded {len(injected_data)} cached injection records')
else:
    injected_data = {}
    no_rbench = []
    print(f'Injecting from R-Bench GT=no questions ({len(captions)} images)...')

    for img_id, cap in captions.items():
        qs = rbench_questions.get(img_id, [])
        no_qs = [qa for qa in qs if qa['answer'].lower().strip() == 'no']
        if not no_qs:
            no_rbench.append(img_id)
            continue

        no_qs_sorted = sorted(no_qs, key=lambda qa: len(qa['question']), reverse=True)
        chosen = no_qs_sorted[0]
        question = chosen['question']

        statement = question_to_statement(question)
        if not statement:
            continue

        sep = ' ' if cap.rstrip().endswith('.') else '. '
        corrupted = cap.rstrip() + sep + statement

        subj, rel, obj_ = parse_question(question)

        injected_data[img_id] = {
            'original_caption':  cap,
            'corrupted_caption': corrupted,
            'injected_question': question,
            'injected_statement': statement,
            'subject':  subj,
            'relation': rel,
            'object':   obj_,
            'rel_type': classify_rel_type(question),
            'gt':       'no',
        }
        print(f'  [{img_id}] Q: {question}')
        print(f'    cap: {cap[:70]}')
        print(f'    inj: {corrupted[:70]}')

    with open(INJECT_PATH, 'w') as f: json.dump(injected_data, f, indent=2)

    if no_rbench:
        print(f'\nNo GT=no questions for {len(no_rbench)} images: {no_rbench[:5]}')

n_inj   = len(injected_data)
n_total = len(captions)
print(f'\nInjected: {n_inj}/{n_total} images ({100*n_inj/max(n_total,1):.0f}%)')
if n_inj == 0:
    print('WARNING: No injections created. Check rbench_questions is populated.')

In [ ]:
# ============================================================
# Cell 5 — RelCheck Detection on Corrupted Captions
# ============================================================
DETECT_PATH = f'{SAVE_DIR}/detection_results_{CAPTIONER}.json'

print(f'Running RelCheck detection on {len(injected_data)} corrupted triples...\n')

detection_results = []
detected = missed = uncertain = 0

for img_id, inj in injected_data.items():
    pil = pil_images.get(img_id)
    if pil is None:
        print(f'  [{img_id}] image not loaded — skipping')
        continue

    s, r, o = inj['subject'], inj['relation'], inj['object']
    rel_type = classify_relation(r)

    # Run detection + verification from relcheck_v2
    all_entities = list(set([s, o]))
    detections = detect_objects(pil, all_entities)
    verdict = verify_triple(s, r, o, detections, pil)

    if verdict is False:
        status = 'DETECTED'
        detected += 1
    elif verdict is True:
        status = 'MISSED'
        missed += 1
    else:
        status = 'UNCERTAIN'
        uncertain += 1

    result = {
        'img_id': img_id,
        'subject': s, 'relation': r, 'object': o,
        'injected_question': inj['injected_question'],
        'rel_type': rel_type,
        'verdict': str(verdict),
        'status': status,
    }
    detection_results.append(result)
    icon = {'DETECTED': '✓', 'MISSED': '✗', 'UNCERTAIN': '?'}[status]
    print(f'  [{img_id}] {icon} {status}: ({s}, {r}, {o}) [{rel_type}]')

with open(DETECT_PATH, 'w') as f:
    json.dump(detection_results, f, indent=2)

total = detected + missed + uncertain
print(f'\n── Detection Results ──────────────────────────────')
print(f'  Total injected:  {total}')
print(f'  DETECTED:        {detected} ({100*detected/max(total,1):.1f}%)')
print(f'  MISSED:          {missed} ({100*missed/max(total,1):.1f}%)')
print(f'  UNCERTAIN:       {uncertain} ({100*uncertain/max(total,1):.1f}%)')
print(f'  Detection recall: {100*detected/max(total,1):.1f}%')

by_type = defaultdict(lambda: {'detected':0, 'missed':0, 'uncertain':0})
for r in detection_results:
    by_type[r['rel_type']][r['status'].lower()] += 1
print(f'\n  By relation type:')
for rtype, counts in sorted(by_type.items()):
    t = sum(counts.values())
    d = counts['detected']
    print(f'    {rtype:12} — {d}/{t} detected ({100*d/max(t,1):.0f}%)')

In [ ]:
# ============================================================
# Cell 6 — Visual Knowledge Base Construction
# ============================================================
KB_PATH = f'{SAVE_DIR}/kb_{CAPTIONER}.json'

if os.path.exists(KB_PATH):
    with open(KB_PATH) as f:
        knowledge_bases = json.load(f)
    print(f'Loaded KB from cache: {KB_PATH} ({len(knowledge_bases)} images)')
else:
    print(f'Building Visual KB for {len(pil_images)} images...')

    knowledge_bases = {}
    for idx, (img_id, img) in enumerate(pil_images.items()):
        t0 = time.time()
        cap = captions.get(img_id, '')

        # Build KB using relcheck_v2 module
        kb = build_visual_kb(img, cap, max_detections=20)
        knowledge_bases[img_id] = kb

        elapsed = time.time() - t0
        n_objs = len(Counter(l for l, _, _ in kb['detections']))
        n_spatial = len(kb['spatial_facts'])
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f'  [{idx+1}/{len(pil_images)}] {img_id}: {n_objs} objs, '
                  f'{n_spatial} spatial, {elapsed:.1f}s')
        time.sleep(0.3)

    with open(KB_PATH, 'w') as f:
        json.dump(knowledge_bases, f)

    avg_objs = sum(len(Counter(l for l,_,_ in kb['detections'])) for kb in knowledge_bases.values()) / max(len(knowledge_bases), 1)
    avg_spatial = sum(len(kb['spatial_facts']) for kb in knowledge_bases.values()) / max(len(knowledge_bases), 1)
    print(f'\nDone. Avg objects/image: {avg_objs:.1f} | Avg spatial facts: {avg_spatial:.1f}')
    print(f'Saved to {KB_PATH}')

In [ ]:
# ============================================================
# Cell 7 — Full RelCheck Correction Pipeline
# ============================================================
# Uses relcheck_v2.correction module for all correction logic.
# The module contains: enrich_caption_v3 (router),
#   _enrich_short_caption, _correct_long_caption_v2,
#   and all supporting functions (VQA, geometry, batch correction).

from relcheck_v2.correction import (
    enrich_caption_v3,
    _correct_long_caption_v2,
    _enrich_short_caption,
)

CORRECTED_PATH = f'{SAVE_DIR}/corrected_{CAPTIONER}.json'

if os.path.exists(CORRECTED_PATH):
    with open(CORRECTED_PATH) as f:
        correction_data = json.load(f)
    corrected_captions = {img_id: d['corrected'] for img_id, d in correction_data.items()}
    print(f'Loaded {len(corrected_captions)} cached corrections from {CORRECTED_PATH}')
else:
    print(f'Running RelCheck correction on {len(injected_data)} corrupted captions...\n')

    correction_data = {}
    corrected_captions = {}

    for idx, (img_id, inj) in enumerate(injected_data.items()):
        t0 = time.time()
        corrupted_cap = inj['corrupted_caption']
        pil = pil_images.get(img_id)
        kb = knowledge_bases.get(img_id, {})

        if pil is None:
            print(f'  [{img_id}] image not loaded — skipping')
            continue

        # For synthetic test: always use surgical correction mode
        # (not enrichment), regardless of caption length
        result = _correct_long_caption_v2(
            corrupted_cap, pil, kb,
            other_captions=None,
        )

        corrected = result.get('corrected', corrupted_cap)
        corrected_captions[img_id] = corrected
        correction_data[img_id] = {
            'original':  inj['original_caption'],
            'corrupted': corrupted_cap,
            'corrected': corrected,
            'errors':    result.get('errors', []),
            'edit_rate': result.get('edit_rate', 0),
            'status':    result.get('status', 'unknown'),
            'mode':      result.get('mode', 'correct'),
        }

        elapsed = time.time() - t0
        changed = corrected != corrupted_cap
        tag = 'CHANGED' if changed else 'same'
        print(f'  [{idx+1}/{len(injected_data)}] [{img_id}] {tag} ({elapsed:.1f}s)')
        if changed:
            print(f'    before: {corrupted_cap[:80]}')
            print(f'    after:  {corrected[:80]}')

        # Checkpoint every 10 images
        if (idx + 1) % 10 == 0:
            with open(CORRECTED_PATH, 'w') as f:
                json.dump(correction_data, f, indent=2)

    with open(CORRECTED_PATH, 'w') as f:
        json.dump(correction_data, f, indent=2)

    n_changed = sum(1 for d in correction_data.values() if d['corrected'] != d['corrupted'])
    print(f'\nDone. {n_changed}/{len(correction_data)} captions modified.')
    print(f'Saved to {CORRECTED_PATH}')

In [ ]:
# ============================================================
# Cell 8 — R-POPE LLM-Judge Evaluation
# ============================================================

corrected_img_ids = {
    img_id for img_id, inj in injected_data.items()
    if img_id in corrected_captions
    and corrected_captions[img_id] != inj['corrupted_caption']
}
print(f'Images modified by RelCheck: {len(corrected_img_ids)}/{len(injected_data)}')

# ── Main evaluation loop ─────────────────────────────────
print('\nRunning R-POPE LLM-Judge on injected questions...\n')

total = orig_no = corr_no = fixed_no = 0
true_drops = recoveries = 0
per_type = {}
rows = []

for img_id, inj in injected_data.items():
    if img_id not in corrected_captions: continue

    question = inj['injected_question']
    rel_type = inj.get('rel_type', '?')
    orig_cap  = inj['original_caption']
    corr_cap  = inj['corrupted_caption']
    fixed_cap = corrected_captions[img_id]

    orig_ans  = rpope_judge(orig_cap,  question)
    corr_ans  = rpope_judge(corr_cap,  question)
    fixed_ans = rpope_judge(fixed_cap, question)

    total += 1
    if orig_ans  == 'no': orig_no  += 1
    if corr_ans  == 'no': corr_no  += 1
    if fixed_ans == 'no': fixed_no += 1

    injected_ok = (orig_ans == 'no' and corr_ans == 'yes')
    if injected_ok:
        true_drops += 1
        if fixed_ans == 'no': recoveries += 1

    pt = per_type.setdefault(rel_type,
         {'total':0,'orig_no':0,'corr_no':0,'fixed_no':0,'drops':0,'rec':0})
    pt['total']   += 1
    if orig_ans  == 'no': pt['orig_no']  += 1
    if corr_ans  == 'no': pt['corr_no']  += 1
    if fixed_ans == 'no': pt['fixed_no'] += 1
    if injected_ok:        pt['drops']   += 1
    if injected_ok and fixed_ans == 'no': pt['rec'] += 1

    det = ' <-- DETECTED' if injected_ok else ''
    rec = ' + RECOVERED' if (injected_ok and fixed_ans == 'no') else ''
    rows.append((img_id, rel_type, orig_ans, corr_ans, fixed_ans, det+rec))
    print(f'  [{img_id}] [{rel_type}]{det}{rec}')
    print(f'    Q: {question}')
    print(f'    orig={orig_ans}  corr={corr_ans}  fixed={fixed_ans}')

# ── Summary ──────────────────────────────────────────────
if total == 0:
    print('WARNING: nothing evaluated — check injected_data and corrected_captions')
else:
    acc_orig  = 100 * orig_no  / total
    acc_corr  = 100 * corr_no  / total
    acc_fixed = 100 * fixed_no / total
    drop     = acc_corr  - acc_orig
    recovery = acc_fixed - acc_corr

    print(f'\n' + '='*60)
    print(f'R-POPE Results (GT=no for all injected questions)')
    print(f'='*60)
    print(f'  Images evaluated:   {total}')
    print(f'  Injection detected: {true_drops}/{total}  ({100*true_drops/total:.0f}%)')
    print(f'  Recoveries:         {recoveries}/{max(true_drops,1)}  ({100*recoveries/max(true_drops,1):.0f}% of detectable)')
    print(f'')
    print(f'  Accuracy (fraction answering no = correct):')
    print(f'    Original:   {acc_orig:.1f}%  (caption without hallucination)')
    print(f'    Corrupted:  {acc_corr:.1f}%  (after injection,   delta={drop:+.1f}%)')
    print(f'    Corrected:  {acc_fixed:.1f}%  (after RelCheck,   delta={recovery:+.1f}%)')
    print(f'')
    print(f'  By relation type:')
    for rtype, c in sorted(per_type.items()):
        t = c['total']
        print(f"    {rtype:10} n={t}  orig={c['orig_no']}/{t}  "
              f"corr={c['corr_no']}/{t}  fixed={c['fixed_no']}/{t}  "
              f"drops={c['drops']}  rec={c['rec']}")

    # ── Supplemental: all OTHER R-Bench questions ──────────
    print(f'\n' + '='*60)
    print(f'Supplemental: all other R-Bench questions (same images)')
    print(f'='*60)
    rb2_total = rb2_orig = rb2_corr = rb2_fixed = 0
    rbc_total = rbc_orig = rbc_corr = rbc_fixed = 0
    for img_id, inj in injected_data.items():
        if img_id not in corrected_captions: continue
        injected_q = inj['injected_question']
        is_corrected = img_id in corrected_img_ids
        for qa in rbench_questions.get(img_id, []):
            if qa['question'] == injected_q: continue
            gt = qa['answer'].lower().strip()
            if gt not in ('yes','no'): continue
            oa = rpope_judge(inj['original_caption'],  qa['question'])
            ca = rpope_judge(inj['corrupted_caption'], qa['question'])
            fa = rpope_judge(corrected_captions[img_id], qa['question'])
            rb2_total += 1
            if oa == gt: rb2_orig  += 1
            if ca == gt: rb2_corr  += 1
            if fa == gt: rb2_fixed += 1
            if is_corrected:
                rbc_total += 1
                if oa == gt: rbc_orig  += 1
                if ca == gt: rbc_corr  += 1
                if fa == gt: rbc_fixed += 1
    if rb2_total > 0:
        s_orig  = 100*rb2_orig /rb2_total
        s_corr  = 100*rb2_corr /rb2_total
        s_fixed = 100*rb2_fixed/rb2_total
        print(f'  All images ({rb2_total} other questions):')
        print(f'    Original:  {s_orig:.1f}%')
        print(f'    Corrupted: {s_corr:.1f}%  (delta={s_corr-s_orig:+.1f}%)')
        print(f'    Corrected: {s_fixed:.1f}%  (delta={s_fixed-s_corr:+.1f}%)')
        if rbc_total > 0:
            bc_orig  = 100*rbc_orig /rbc_total
            bc_corr  = 100*rbc_corr /rbc_total
            bc_fixed = 100*rbc_fixed/rbc_total
            n_corr = len(corrected_img_ids & set(injected_data))
            print(f'  Corrected images only (n={n_corr}, {rbc_total} questions):')
            print(f'    Original:  {bc_orig:.1f}%')
            print(f'    Corrupted: {bc_corr:.1f}%  (delta={bc_corr-bc_orig:+.1f}%)')
            print(f'    Corrected: {bc_fixed:.1f}%  (delta={bc_fixed-bc_corr:+.1f}%)')
    else:
        print('  No supplemental questions found.')